In [2]:
import pandas as pd
import numpy as np

## Reading Data in Text Format

pandas provides a set of functions for reading tabular text data into a DataFrame.

### Common Reading Functions

| Function | Description |
|----------|-------------|
| `read_csv` | Comma-delimited files (default delimiter is `,`) |
| `read_fwf` | Fixed-width column format (no delimiter) |
| `read_clipboard` | Like `read_csv` but reads from the clipboard |
| `read_excel` | Excel XLS or XLSX files |
| `read_html` | All tables found in an HTML document |
| `read_json` | JSON string, file, or URL |
| `read_parquet` | Apache Parquet binary format |
| `read_sql` | Results of a SQL query (via SQLAlchemy) |
| `read_stata` | Stata file format |
| `read_xml` | XML file |

Binary formats (`read_hdf`, `read_feather`, `read_orc`, `read_pickle`, `read_parquet`) have data type information embedded — no type inference needed.

### Categories of Optional Arguments

- **Indexing** — which column(s) to use as the row index, where to get column names.
- **Type inference and data conversion** — custom value converters, missing value markers.
- **Date and time parsing** — parsing and combining date/time columns.
- **Iteration** — reading very large files in chunks.
- **Unclean data** — skipping rows, handling comments, thousands separators.

`read_csv` has around 50 parameters — consult the pandas documentation for specific use cases.

## Key Points

- `read_csv` is the most commonly used text data reader.
- Text formats require type inference; binary formats (Parquet, HDF5, ORC) carry type info.
- Optional arguments fall into five categories: indexing, type conversion, date parsing, iteration, and unclean data.

## Basic CSV Reading

### With a Header Row

By default `read_csv` uses the first row as column names and assigns an integer index.

### Without a Header Row

If the file has no header:

- `header=None` → pandas assigns numeric column names (`0, 1, 2, ...`).
- `names=[...]` → supply custom column names directly.

### Setting a Column as the Index

Use `index_col` to designate one or more columns as the row index:

- Single column: `index_col="message"` or `index_col=4`.
- Multiple columns: `index_col=["key1", "key2"]` → creates a **hierarchical (MultiIndex)**.

### Variable Whitespace Delimiter

When fields are separated by variable amounts of whitespace, pass a regular expression as `sep`:

```python
pd.read_csv(file, sep=r"\s+")
```

If there is one fewer column name than data columns, pandas infers the first column is the index.

### Skipping Rows

Use `skiprows` with a list of row numbers (0-based) to skip specific rows such as comments:

```python
pd.read_csv(file, skiprows=[0, 2, 3])
```

## Key Points

- Default: first row is header, integer row index assigned automatically.
- `header=None` for files with no header; `names=` to supply column names.
- `index_col` sets one column (or a list of columns) as the row index.
- `sep=r"\s+"` handles variable-whitespace-delimited files.
- `skiprows=[...]` skips specific rows by 0-based row number.

In [3]:
# ex1.csv has a header row
df = pd.read_csv("../examples/ex1.csv")

df

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [4]:
# ex2.csv has no header — let pandas assign numeric column names
pd.read_csv("../examples/ex2.csv", header=None)

,0,1,2,3,4
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [5]:
# Supply column names manually
pd.read_csv("../examples/ex2.csv", names=["a", "b", "c", "d", "message"])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [6]:
# Set a column as the row index
names = ["a", "b", "c", "d", "message"]

pd.read_csv("../examples/ex2.csv", names=names, index_col="message")

,a,b,c,d
message,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


In [7]:
# Hierarchical index from multiple columns
pd.read_csv("../examples/csv_mindex.csv", index_col=["key1", "key2"])

value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16

In [8]:
# Variable whitespace separator — use regex \s+
pd.read_csv("../examples/ex3.txt", sep=r"\s+")

,A,B,C
aaa,-0.264438,-1.026059,-0.619500
bbb,0.927272,0.302904,-0.032399
ccc,-0.264273,-0.386314,-0.217601
ddd,-0.871858,-0.348382,1.100491


In [9]:
# Skip specific rows by index (0-based)
pd.read_csv("../examples/ex4.csv", skiprows=[0, 2, 3])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


## Handling Missing Values

Missing data in a file can appear as:

- An **empty field** (nothing between two delimiters).
- A **sentinel string** like `NA`, `NULL`, `N/A`, etc.

pandas recognizes a default set of sentinel strings and converts them to `NaN`.

### Customising NA Detection

| Argument | Effect |
|----------|--------|
| `na_values=[...]` | Add strings to the default NA list |
| `keep_default_na=False` | Disable all default NA sentinels |
| `na_values={col: [...]}` | Specify NA sentinels per column using a dictionary |

Combining `keep_default_na=False` with `na_values=[...]` gives full control: only the strings you list will be treated as missing.

Use `pd.isna(df)` or `df.isna()` to inspect which cells were parsed as missing.

## Key Points

- pandas has a built-in list of NA sentinels (`NA`, `NULL`, empty string, etc.) applied by default.
- `na_values` adds to the default list; it does not replace it unless `keep_default_na=False`.
- `keep_default_na=False` disables all defaults — only explicitly listed values become `NaN`.
- A dictionary passed to `na_values` sets per-column NA sentinels.

In [10]:
result = pd.read_csv("../examples/ex5.csv")

result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [11]:
pd.isna(result)

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,True,False,False
2,False,False,False,False,False,False


In [12]:
# Add "NULL" to the NA sentinel list
pd.read_csv("../examples/ex5.csv", na_values=["NULL"])

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [13]:
# Disable default NA sentinels — nothing is treated as missing
result2 = pd.read_csv("../examples/ex5.csv", keep_default_na=False)

result2

,something,a,b,c,d,message
0,one,1,2,3,4,NA
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [14]:
result2.isna()

,something,a,b,c,d,message
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False


In [15]:
# Disable defaults, explicitly mark only "NA" as missing
result3 = pd.read_csv("../examples/ex5.csv", keep_default_na=False, na_values=["NA"])

result3

,something,a,b,c,d,message
0,one,1,2,3,4,NaN
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [16]:
result3.isna()

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,False,False,False
2,False,False,False,False,False,False


In [17]:
# Per-column NA sentinels via a dictionary
sentinels = {"message": ["foo", "NA"], "something": ["two"]}

pd.read_csv("../examples/ex5.csv", na_values=sentinels, keep_default_na=False)

,something,a,b,c,d,message
0,one,1,2,3,4,NaN
1,NaN,5,6,,8,world
2,three,9,10,11,12,NaN


## `read_csv` Argument Reference

| Argument | Description |
|----------|-------------|
| `path` | Filesystem path, URL, or file-like object |
| `sep` / `delimiter` | Character or regex to split fields; default `","` |
| `header` | Row number to use as column names; `None` if no header |
| `index_col` | Column(s) to use as the row index; list for hierarchical index |
| `names` | List of column names for the result |
| `skiprows` | Number of rows to skip at the start, or list of row numbers to skip |
| `na_values` | Additional strings to recognise as `NaN` |
| `keep_default_na` | Use default NA sentinel list; `True` by default |
| `comment` | Character(s) that mark comments at end of lines |
| `parse_dates` | Attempt to parse columns as datetimes; `False` by default |
| `keep_date_col` | Keep original columns after joining them for date parsing; `False` by default |
| `converters` | Dict mapping column name/number to a conversion function |
| `dayfirst` | Treat ambiguous dates as day-first (international format); `False` by default |
| `nrows` | Number of rows to read from the beginning of the file |
| `iterator` | Return a `TextFileReader` for reading the file piecemeal |
| `chunksize` | Number of rows per chunk when iterating |
| `skip_footer` | Number of lines to ignore at the end of the file |
| `encoding` | Text encoding (e.g. `"utf-8"`); defaults to `"utf-8"` |
| `thousands` | Thousands separator character (e.g. `","` or `"."`) |
| `decimal` | Decimal separator character; default `"."` |
| `engine` | CSV parser engine: `"c"` (default), `"python"`, or `"pyarrow"` (fastest for large files) |